## Formulario correo

In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [7]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)


In [19]:
filename='color_alfin.xlsx'
ruta_archivo = os.path.join(ruta_csv, filename)
df_target_desembolso = pd.read_excel(ruta_archivo, sheet_name='Hoja2')

In [20]:
df_target_desembolso["color_tono"] = (df_target_desembolso["Color"] + " " + df_target_desembolso["Tono"]).str.upper()

In [18]:
print(df_target_desembolso.columns.to_list())

['AñoMes', 'fecha_desembolso', 'CanalBT', 'CanalVenta', 'Dim_Agencias_mapeo.Region', 'Cod_Sucursal_Origen', 'Sucursal_Origen', 'cod_sucursal_verificado', 'Sucursal', 'cod_colaborador', 'cod_colaborador_verificado', 'Dim_ANC_Mapeo_RRHH (5).NomEmp', 'VendorBT', 'vendor_verificado', 'CuentaCliente', 'Operacion', 'Nombres', 'moneda', 'CapitalSolicitado', 'TotalCapital', 'TotalSeguros', 'VencimientoPrimeraCuota', 'VencimientoOperacion', 'ValorCuota', 'CantidadCuotas', 'tipo_producto', 'Tasa_Mixta', 'TasaEfectivaAnual', 'TEA_1', 'TEA_2', 'TCEA', 'Campania', 'Campaña_Grupo', 'Color', 'Tono', 'nom_user_v3', 'grupo_user_v3', 'TipoCliente', 'TipoClienteRiegsos', 'Oferta_max', 'EstadoCredito', 'Seg_familia', 'Seg_dp', 'codigo_id', 'color_tono']


In [27]:
df_target_desembolso=df_target_desembolso[['fecha_desembolso','color_tono','NumeroDcumento','Oferta_max','Dim_ANC_Mapeo_RRHH (5).NomEmp','Dim_Agencias_mapeo.Region','Cod_Sucursal_Origen','Sucursal_Origen','cod_sucursal_verificado']]

In [29]:
df_target_desembolso = df_target_desembolso.drop_duplicates(
    subset=['NumeroDcumento'],
    keep='first'
)

In [32]:
df_target_desembolso.rename(columns={'NumeroDcumento': 'dni_cliente'}, inplace=True)
# df_target_desembolso.rename(columns={'color_tono': 'color'}, inplace=True)
# df_target_desembolso.rename(columns={'Oferta_max': 'monto'}, inplace=True)

df_target_desembolso['dni_cliente'] = (
    df_target_desembolso['dni_cliente']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)


In [34]:
ruta_archivo = os.path.join(ruta_csv, 'tmp_muestra_alfin.csv')
df_target_desembolso['dni_cliente'].to_csv(ruta_archivo, index=False,sep=';')

### spark -- datos faltantes

In [2]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [71]:
filename='Consulta_de_Campañas_202607_V2_SS_EXT_CAMBIO_db.csv'
df_validar_01=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202607_V2_SS_EXT_CAMBIO_db2.csv'
df_validar_02=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
df_validar=df_validar_01.unionByName(df_validar_02)

filename='tmp_muestra_alfin.csv'
df_tmp_dni=cargar_archivo_csv(spark,filename,';',True)
print(df_validar.columns)
print(df_tmp_dni.columns)


['DNI', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TASA_MIN_DESCUENTO', 'TIPO']
['dni_cliente']


In [73]:

df_tmp_dni = df_tmp_dni.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("dni_cliente")),
        F.lit(8)
    )
)

In [ ]:
filename='Libro2.csv'
df_usar=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_csv)

df_usar = df_usar.withColumn(
    "DNI",
    F.right(
        F.concat(F.lit("00000000"), F.col("DNI")),
        F.lit(8)
    )
)
overwrite_table_SQL(spark,df_usar,f'ASSSSSSSSS_BOORAR',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [12]:

df_usar = df_usar.withColumn(
    "DNI",
    F.right(
        F.concat(F.lit("00000000"), F.col("DNI")),
        F.lit(8)
    )
)
overwrite_table_SQL(spark,df_usar,f'ASSSSSSSSS_BOORAR',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [67]:

df_usar = df_usar.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("dni_cliene")),
        F.lit(8)
    )
)

In [74]:
df_validar=df_validar.withColumnRenamed('DNI','dni_cliente')
df_validar=df_validar.join(df_tmp_dni,['dni_cliente'],'inner')

In [ ]:
    query = f"""
        SELECT DISTINCT dni_cliente,contacto,retiro4
        FROM (
            SELECT dni_cliente,celular as contacto,
            case
                when tipificacion in (4,5) then 'seguimiento -1'
                else 'no llamar -1'
            end as retiro4
            FROM valentina.dbo.alfin_gestion
            WHERE DNI_ejecutivo!='99999999' 
            and tipificacion in ('4','5','16','17')
            and CAST(fecha AS DATE) >= DATEADD(MONTH, -1, CAST('{fecha_mes_base}' AS DATE))
            AND CAST(fecha AS DATE) < DATEADD(MONTH, 0, CAST('{fecha_mes_base}' AS DATE))
            UNION ALL
            SELECT DISTINCT dni_cliente,CELULAR as contacto, 'excluir' as retiro4
            FROM valentina.dbo.Excluir_Gestion_TARGET where servicio in ('ALFIN','ALFCC')
        ) t
        """
    df_retiro_dni_contacto=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)

In [5]:
query = """

Select a.*,
b.TIPO as Estado_,
b.SUB_DESCRIPCION as Sub_Estado_,
b.DESCRIPCION as Descripcion_,
b.[PESO] as Pesos,
a.dni+' '+a.PHONE_NUMBER AS Enlace,
a.Fecha_Llamada as Fecha_Llam,
DATEPART(hh,a.Fecha_Hora_Llamada) as Hora_Llamada,
[RH]=case when Trama_Hora >= 15 then 3 when Trama_Hora >= 12 then 2 else 1 END,
b.[COD_BCO] as COD_BCO
From SAMANTHA.dbo.tmp_llamadas_mes a WITH (NOLOCK) 
LEFT JOIN ODIN.[dbo].[tTipologia_Cencosud_PPFF] b WITH (NOLOCK) on a.Codigo_Paleta=b.Codigo
where a.Fecha_Llamada >='2026-06-01'
AND A.dni='09271866'

    """
df_cenco=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

In [6]:
df_cenco.count()

42

In [7]:
df_dni=df_cenco.toPandas()

In [8]:

ruta_archivo = os.path.join(ruta_csv, 'solo_09271866_cenco.csv')

df_dni.to_csv(ruta_archivo, sep=';')

In [68]:
df_usar=df_usar.select('dni_cliente','celular')

In [69]:
df_usar=df_base.unionByName(df_usar)

In [ ]:
query = """
    SELECT DISTINCT dni_cliente,contacto,retiro4
    FROM (
        select NUMERO_DOCUMENTO as dni_cliente, cl_telf1 as celular
        from DANTALION.dbo.Base_Maestra_ALFIN_BK_Vigente
        UNION 
        select NUMERO_DOCUMENTO as dni_cliente, cl_telf1 as celular
        from DANTALION.dbo.Base_Maestra_ALFIN_BK_Vigente
        ) t
    """
df_base=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

In [ ]:
filename='usar_alfin.csv'
df_usar=cargar_archivo_csv(spark,filename,';',True)

In [75]:
df_validar.join(df_usar,['dni_cliente'],'inner').count()


306

In [41]:

df_validar.join(df_tmp_dni,['dni_cliente'],'inner').count()

600

In [ ]:
filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()

# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)

dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())

df_target_desembolso['retiro'] = (
    df_target_desembolso['dni_cliente'].isin(dni_retiro) |
    df_target_desembolso['celular'].isin(cel_retiro)
).astype(int)

In [190]:

df_tmp_dni = df_tmp_dni.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("dni_cliente")),
        F.lit(8)
    )
)

df_validar = df_validar.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("DNI")),
        F.lit(8)
    )
).drop('DNI')

In [191]:
# df_tmp_retiro = df_tmp_retiro.withColumn(
#     "dni_cliente",
#     F.right(
#         F.concat(F.lit("00000000"), F.col("dni_cliente")),
#         F.lit(8)
#     )
# ).drop('DNI')

In [192]:
overwrite_table_SQL(spark,df_tmp_dni,f'tmp_muestra_cliente_alfin_borrar',server_kishin,user_kishin,pwd_kishin,'DANTALION')
# overwrite_table_SQL(spark,df_tmp_retiro,f'tmp_muestra_cliente_alfin_borrar_retiro_1',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [193]:
query = """
    select * from DANTALION.dbo.tmp_muestra_cliente_alfin_borrar 
    """
df_dni_ref=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

df_validar=df_validar.join(df_dni_ref,['dni_cliente'],'inner')

query = """
    select a.NUMERO_DOCUMENTO as dni_cliente
    from DANTALION.dbo.Base_Maestra_ALFIN_BK_Vigente a
    inner join tmp_muestra_cliente_alfin_borrar b
    on a.NUMERO_DOCUMENTO=b.dni_cliente
    """
df_ref_base=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

df_validar_pd = df_validar.toPandas()
df_ref_base_pd = df_ref_base.toPandas()

### completar base + columnas de consulta campaña

In [194]:
dni_en_base = set(df_ref_base_pd['dni_cliente'].dropna())

df_seguimiento_1['en_base'] = (
    df_seguimiento_1['dni_cliente'].isin(dni_en_base) 
).astype(int)

In [195]:
df = df_seguimiento_1.copy()

# Asegurar que dia_ref sea fecha
df['dia_ref'] = pd.to_datetime(df['dia_ref'], errors='coerce')

# Cantidad de envíos por DNI
df['q_envios'] = df.groupby('dni_cliente')['dni_cliente'].transform('size')

df_resumen = (
    df.sort_values('dia_ref')
      .drop_duplicates(subset='dni_cliente', keep='last')
      .copy()
)

import numpy as np

hoy = np.datetime64('today', 'D')

df_resumen['conteo_dia'] = np.busday_count(
    df_resumen['dia_ref'].values.astype('datetime64[D]'),
    hoy,
    weekmask='1111110'   # Lunes a sábado
)

# Crear env_1 hasta env_6
for i in range(1, 7):
    df_resumen[f'env_{i}'] = (df_resumen['q_envios'] == i).astype(int)

In [196]:
df_validar_pd_seleccion=df_validar_pd[['dni_cliente','COLOR_FINAL','USER_V3','FRESCURA','PROPENSION_DISTRIBUCION','OFERTA_MAX']]

In [211]:
df_resumen_1=df_resumen.merge(df_validar_pd_seleccion,on='dni_cliente',how='left')
df_resumen_1 = df_resumen_1.drop_duplicates(subset=['dni_cliente'])


In [212]:
ruta_archivo = os.path.join(ruta_csv, 'tmp_resumen1.csv')
df_resumen_1.to_csv(ruta_archivo, index=False,sep=';')

In [46]:
query = f"""
	SELECT * FROM Alice.prospectos_correos_alfin 
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)
ruta_archivo = os.path.join(ruta_csv, 'revisar.csv')
df_prospectos_correos_alfin.to_csv(ruta_archivo, index=False,sep=';')

In [ ]:
['dni_cliente', 'dia_ref', 'celular', 'target', 'MONTO', 'ASESOR', 'CANALVENTA', 'fugas', 'retiro', 'en_base', 'q_envios', 'conteo_dia', 'env_1', 'env_2', 'env_3', 'env_4', 'env_5', 'env_6', 'color', 'USER_V3', 'FRESCURA', 'PROPENSION_DISTRIBUCION', 'OFERTA_MAX']

['dni_cliente', 'dia_ref', 'celular', 'target', 'MONTO', 'ASESOR', 'CANALVENTA', 'fugas', 'retiro', 'en_base', 'q_envios', 'conteo_dia', 'env_1', 'env_2', 'env_3', 'env_4', 'env_5', 'env_6', 'color', 'USER_V3', 'FRESCURA', 'PROPENSION_DISTRIBUCION', 'OFERTA_MAX']


In [ ]:
fechas = pd.to_datetime(['2026-07-16', '2026-07-17', '2026-07-18'])

df_resumen_1=df_resumen_1[
    (df_resumen_1['fugas'] == 0) &
    (df_resumen_1['retiro'] == 0) &
    (df_resumen_1['dia_ref'].isin(fechas)) &
    (df_resumen_1['MONTO'].isna()) &
    (df_resumen_1['q_envios'] > 1)
]

In [214]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)


In [215]:
df_resumen_1[df_resumen_1['MONTO'].isna()].head()

,dni_cliente,dia_ref,celular,target,MONTO,ASESOR,CANALVENTA,fugas,retiro,en_base,q_envios,conteo_dia,env_1,env_2,env_3,env_4,env_5,env_6,COLOR_FINAL,USER_V3,FRESCURA,PROPENSION_DISTRIBUCION,OFERTA_MAX
3666,25012211,2026-07-16,904204204,0,NaN,NaN,NaN,0,0,0,2,3,0,1,0,0,0,0,VERDE OSCURO,3. MES + PLD Peers,2,1,8200
3683,23826476,2026-07-16,987478747,0,NaN,NaN,NaN,0,0,1,2,3,0,1,0,0,0,0,VERDE OSCURO,3. MES + PLD Peers,0,1,10000
3691,23814493,2026-07-16,952394524,0,NaN,NaN,NaN,0,0,0,2,3,0,1,0,0,0,0,NARANJA CLARO,7. Peers,0,1,3000
3695,23808230,2026-07-16,942210932,0,NaN,NaN,NaN,0,0,0,2,3,0,1,0,0,0,0,AMARILLO CLARO,1. sunedu & sunarp A,0,2,13700
3717,23927174,2026-07-16,984270438,0,NaN,NaN,NaN,0,0,0,2,3,0,1,0,0,0,0,VERDE CLARO,3. MES + PLD Peers,3,3,7700


In [216]:
df_resumen_1=df_resumen_1.rename(columns={'COLOR_FINAL':'color'})

In [162]:
df_resumen_1.shape

(890, 23)

In [156]:
df_prospectos_correos_alfin=df_prospectos_correos_alfin.drop(columns='color')


In [233]:
df_prospectos_correos_alfin=df_prospectos_correos_alfin.drop(columns='color')
df_prospectos_envio_alfin=df_prospectos_envio_alfin.merge(df_resumen_1[['dni_cliente','color']],on='dni_cliente',how='inner')
df_prospectos_correos_alfin=df_prospectos_correos_alfin.merge(df_resumen_1[['dni_cliente','color']],on='dni_cliente',how='inner')

In [234]:
df_prospectos_envio_alfin = df_prospectos_envio_alfin.drop_duplicates(subset=['dni_cliente'])
df_prospectos_correos_alfin = df_prospectos_correos_alfin.drop_duplicates(subset=['dni_cliente'])


In [231]:
df_prospectos_correos_alfin.head()

,id,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color_x,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,fecha_envio,intentos_realizados,estado,tipo_carga,fecha_registro,fecha_dia,color_y
0,1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONORE,19337433,JUANA ESTHER VERA MURILLO DE ZAVALETA,VERDE OSCURO,4000.0,963517955,TRUJILLO CENTRO,2026-07-04,0 days 02:06:01,2026-07-03 11:09:44,2,ENVIADO,AUTOMATICO,2026-07-03 02:06:01,2026-07-03,VERDE OSCURO
1,2,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONORE,19998396,ANGEL LUIS PONCE HUARINGA,AMARILLO OSCURO,14000.0,937474638,HUANCAYO,2026-07-04,0 days 02:06:01,2026-07-03 11:14:54,1,ENVIADO,AUTOMATICO,2026-07-03 02:06:01,2026-07-03,AMARILLO OSCURO
2,3,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONORE,40949266,ADA ESMERALDA AYALA HERRERA,AMARILLO OSCURO,3000.0,943587132,AREQUIPA PAMPILLA,2026-07-04,0 days 02:06:02,2026-07-03 11:14:18,1,ENVIADO,AUTOMATICO,2026-07-03 02:06:02,2026-07-03,AMARILLO OSCURO
3,4,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONORE,40319405,ROBERTO SEGUNDO ZEVALLOS CERPA,VERDE OSCURO,9900.0,923809089,AREQUIPA PAMPILLA,2026-07-04,0 days 02:06:02,2026-07-03 11:14:22,1,ENVIADO,AUTOMATICO,2026-07-03 02:06:02,2026-07-03,VERDE OSCURO
4,5,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONORE,00809150,ROGER VALLES RODRIGUEZ,NARANJA CLARO,9800.0,987302232,TARAPOTO,2026-07-04,0 days 02:06:03,2026-07-03 11:14:58,1,ENVIADO,AUTOMATICO,2026-07-03 02:06:03,2026-07-03,NARANJA CLARO


In [220]:
df_prospectos_envio_alfin=df_prospectos_envio_alfin.drop(columns='color')

In [235]:
# df_prospectos_envio_alfin=df_prospectos_envio_alfin.merge(df_resumen_1[['dni_cliente','COLOR_FINAL']],on='dni_cliente',how='inner')
# df_prospectos_correos_alfin=df_prospectos_correos_alfin.merge(df_resumen_1[['dni_cliente','COLOR_FINAL']],on='dni_cliente',how='inner')

df_prospectos_correos_alfin=df_prospectos_correos_alfin[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'color', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita']] .copy()
df_prospectos_correos_alfin['tipo_carga']='MANUAL'

df_prospectos_envio_alfin=df_prospectos_envio_alfin[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()
df_prospectos_correos_alfin['fecha_visita']='2026-07-20'
df_prospectos_envio_alfin['fecha_visita']='2026-07-20'

df_prospectos_correos_alfin = df_prospectos_correos_alfin.drop_duplicates(subset='dni_cliente')
df_prospectos_envio_alfin = df_prospectos_envio_alfin.drop_duplicates(subset='dni_cliente')

display(df_prospectos_correos_alfin.head(2))
display(df_prospectos_envio_alfin.head(2))


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,tipo_carga
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONORE,19337433,JUANA ESTHER VERA MURILLO DE ZAVALETA,VERDE OSCURO,4000.0,963517955,TRUJILLO CENTRO,2026-07-20,0 days 02:06:01,MANUAL
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONORE,19998396,ANGEL LUIS PONCE HUARINGA,AMARILLO OSCURO,14000.0,937474638,HUANCAYO,2026-07-20,0 days 02:06:01,MANUAL


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,00000001,TARGET,19337433,JUANA ESTHER VERA MURILLO DE ZAVALETA,963517955,734265 - TRUJ CENTRO,2026-07-20,4000,Derivacion
1,00000001,TARGET,19998396,ANGEL LUIS PONCE HUARINGA,937474638,734280 - PC HUANCAYO,2026-07-20,14000,Derivacion


In [236]:
df_prospectos_correos_alfin.shape

(1849, 14)

In [237]:


df_prospectos_correos_alfin.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_prospectos_envio_alfin.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

C:\Users\DATA\AppData\Local\Temp\ipykernel_11576\3762344665.py:1: UserWarning: the 'timedelta' type is not supported, and will be written as integer values (ns frequency) to the database.
  df_prospectos_correos_alfin.to_sql(


1849

In [164]:
query = f"""
		SELECT dni_cliente FROM Alice.prospectos_envio_alfin 
    where fecha_visita='2026-07-20'
"""
df_estan = pd.read_sql(query, engine_mysql)

In [165]:
df_resumen_1 = df_resumen_1[
    ~df_resumen_1['dni_cliente'].isin(df_estan['dni_cliente'])
].copy()
df_resumen_1.shape

(0, 23)